In [0]:
dbutils.widgets.dropdown("environment", "dev", ["dev", "prod"])
dbutils.widgets.dropdown("run_mode", "full", ["full", "incremental"])

env = dbutils.widgets.get("environment")
print(env)
run_mode = dbutils.widgets.get("run_mode")
print(run_mode)

In [0]:
%run /Workspace/Users/yash.karda.yk@gmail.com/commerce-data-platform/utils/data_quality.py

In [0]:
%run /Workspace/Users/yash.karda.yk@gmail.com/commerce-data-platform/utils/helpers.py

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
       StructType,StructField,
       StringType,
       IntegerType,
       LongType,
       TimestampType,
       DoubleType
       )

configs = get_config(env)
print(configs)
base_path = configs["base_path"]
layers = configs["layers"]


In [0]:
## Defining Paths
RAW_PATH    = f"{base_path}/raw"
BRONZE_PATH = f"{base_path}/bronze"
SILVER_PATH = f"{base_path}/silver"
GOLD_PATH   = f"{base_path}/gold"
AUDIT_PATH  = f"{base_path}/audit"

print(f"RAW    → {RAW_PATH}")
print(f"BRONZE → {BRONZE_PATH}")
print(f"SILVER → {SILVER_PATH}")
print(f"GOLD   → {GOLD_PATH}")
print(f"AUDIT  → {AUDIT_PATH}")

In [0]:
from datetime import datetime
start_time = datetime.now()
total_rows = 0

df_silver = read_delta(spark, f"{SILVER_PATH}/order_details")
print(f"Silver table loaded: {df_silver.count()} rows")

In [0]:
from pyspark.sql.window import Window

# 1. Revenue by category
df_revenue_by_category = df_silver \
    .groupBy("product_category_name_english") \
    .agg(F.round(F.sum("payment_value"), 2).alias("revenue")) \
    .orderBy("revenue", ascending=False)

# 2. Monthly revenue
df_monthly_revenue = df_silver \
    .withColumn("year", F.year("order_purchase_timestamp")) \
    .withColumn("month", F.month("order_purchase_timestamp")) \
    .groupBy("year", "month") \
    .agg(F.round(F.sum("payment_value"), 2).alias("revenue")) \
    .orderBy("year", "month")

# 3. Top 10 sellers
window_spec = Window.orderBy(F.col("revenue").desc())
df_top_sellers = df_silver \
    .groupBy("seller_id") \
    .agg(F.round(F.sum("payment_value"), 2).alias("revenue")) \
    .withColumn("rank", F.rank().over(window_spec)) \
    .filter(F.col("rank") <= 10)

# 4. Customer LTV
df_customer_ltv = df_silver \
    .groupBy("customer_id") \
    .agg(
        F.count("order_id").alias("order_count"),
        F.round(F.sum("payment_value"), 2).alias("total_revenue"),
        F.round(F.avg("payment_value"), 2).alias("avg_order_value")
    )

# 5. Delivery by state
df_delivery_by_state = df_silver \
    .withColumn("delivery_days", F.datediff(
        F.col("order_delivered_customer_date"),
        F.col("order_purchase_timestamp")
    )) \
    .groupBy("customer_state") \
    .agg(F.round(F.avg("delivery_days"), 2).alias("avg_delivery_days")) \
    .orderBy("avg_delivery_days")

# 6. On time delivery
df_ontime_delivery = df_silver \
    .withColumn("on_time", F.when(
        F.col("order_delivered_customer_date") <= F.col("order_estimated_delivery_date"), 1
    ).otherwise(0)) \
    .groupBy("product_category_name_english") \
    .agg(F.round(F.avg("on_time"), 4).alias("ontime_percentage"))

print("All gold aggregations complete!")

In [0]:
gold_tables = {
    "revenue_by_category": df_revenue_by_category,
    "monthly_revenue": df_monthly_revenue,
    "top_sellers": df_top_sellers,
    "customer_ltv": df_customer_ltv,
    "delivery_by_state": df_delivery_by_state,
    "ontime_delivery": df_ontime_delivery
}

for table_name, df in gold_tables.items():
    write_delta(df, f"{GOLD_PATH}/{table_name}", "overwrite")
    total_rows += df.count()
    print(f"{table_name} written successfully")

In [0]:
end_time = datetime.now()

log_run(
    spark=spark,
    notebook_name="job_03_aggregate",
    environment=env,
    run_mode=run_mode,
    start_time=start_time,
    end_time=end_time,
    rows_processed=total_rows,
    status="SUCCESS",
    audit_path=AUDIT_PATH,
    error_message=None
)

print("Run logged successfully!")

dbutils.notebook.exit("SUCCESS")

In [0]:
spark.read.format("delta").load(f"{AUDIT_PATH}/pipeline_runs").display()